<div style="display: flex; flex-direction: row; justify-content: space-between; font-weight: bold; text-decoration: none">
<a href="../README.md"><<< Back to README File (Introduction)</a>
<a href="02_exploratory_data_analysis.ipynb">Up Next: Exploratory Data Analysis (EDA) >>></a>
</div>

<div style="display: flex; flex-direction: column; align-items: center; font-weight: bold">Want to read a longer version of this file? <a href="01a_short_data_cleaning.ipynb"> Go to Longer Version </a></div>

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Electric Vehicle Data Cleaning and Preparation

## Dataset Sources

**Electric Vehicle Population Dataset from Data.gov** 

- Link to Dataset: https://catalog.data.gov/dataset/electric-vehicle-population-data
- Uses the Open Database License (ODbL): https://opendatacommons.org/licenses/odbl/1-0/

**Gas Prices Dataset**

- Link to Dataset: https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=pet&s=emm_epm0u_pte_swa_dpg&f=a
- Is in the public domain, provided by the U.S. Energy Information Administration (EIA).

Both of these datasets were cleaned and modified during the dataset cleaning sections of this analysis.

## Data Loading

In [3]:
df_gas = pd.read_excel("../data/raw/Gas_Prices.xls", sheet_name="Data 1", header=2)
df_EVs = pd.read_csv("../data/raw/Electric_Vehicle_Population_Data.csv")

## Dataset Information and Notes for Data Cleaning

### Row and Column Info of Gas Prices Dataset - 1st Dataset

In [4]:
print("Row and Column Count:", df_gas.shape)
print("Attribute/Column Names:", df_gas.columns)

Row and Column Count: (23, 2)
Attribute/Column Names: Index(['Date', 'Washington All Grades Conventional Retail Gasoline Prices (Dollars per Gallon)'], dtype='object')


In [5]:
df_gas.head()

,Date,Washington All Grades Conventional Retail Gasoline Prices (Dollars per Gallon)
0,2003-06-30,1.684
1,2004-06-30,1.991
2,2005-06-30,2.410
3,2006-06-30,2.751
4,2007-06-30,3.009


### What to Clean in This Table
**Columns to Rename**

The "Washington All Grades Conventional Retail Gasoline Prices (Dollars per Gallon)" column should something shorter.

**Cell Values to Alter**

If the month and day is the same every row in the Date column, and the year is the only number that changes, we can rewrite the column with just year values in each cell.

### Row and Column Info of Electric Vehicle Population Dataset - 2nd Dataset

In [6]:
print("Row and Column Count:", df_EVs.shape)
print("Attribute/Column Names:", df_EVs.columns)

Row and Column Count: (279780, 16)
Attribute/Column Names: Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Legislative District', 'DOL Vehicle ID', 'Vehicle Location',
       'Electric Utility', '2020 Census Tract'],
      dtype='object')


In [7]:
df_EVs.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,5UX43EU09S,Kitsap,Poulsbo,WA,98370.0,2025,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,40.0,23.0,267525737,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,5.303594e+10
4,3C3CFFGE5F,Thurston,Yelm,WA,98597.0,2015,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87.0,2.0,474468501,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10


### What to Clean in This Table
**Columns to Potentially Drop**
- Candidates for Index Column - I'd need to check if these have unique values. The number of unique values and total rows need to match.
    - VIN (1-10)
    - DOL Vehicle ID
- Legislative District
- 2020 Census Tract

**Column Names**
- CAFV Eligibility: Change column name so that it is phrased as a question, this makes it suited for boolean values in the column.
- Electric Range: Specify the unit of Electric Range, in this case miles.

**Cell Values to Alter**
- Casting to Integer
    - Postal Codes
    - Electric Range
- Electric Vehicle Type: Values can be shortened to just Battery Electric or Plug-in Hybrid.
- CAFV Eligibility: Can be changed to boolean values if column name is phrased as a question.
- Electric Utility: Check to see if there are cell values that are complex. If the cell values are complex, leave them alone.


## Data Cleaning

#### Gas Prices

**Checking first few rows of table**

In [ ]:
df_gas.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,5UX43EU09S,Kitsap,Poulsbo,WA,98370.0,2025,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,40.0,23.0,267525737,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,5.303594e+10
4,3C3CFFGE5F,Thurston,Yelm,WA,98597.0,2015,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87.0,2.0,474468501,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10


**Changing column name from "Washington All Grades Conventional Retail Gasoline Prices (Dollars per Gallon)" to "WA Retail Gasoline Prices (Dollars Per Gallon)"**

In [8]:
df_gas.rename(columns={'Washington All Grades Conventional Retail Gasoline Prices (Dollars per Gallon)': 'WA Retail Gas Prices (Dollars per Gallon)'}, inplace=True)

**Changing Full Dates to Only Year**

In [9]:
df_gas['Date'] = df_gas['Date'].dt.year

**Checking Data Types In Table and Cleaned Version**

In [ ]:
df_gas.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 2 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Date                                       23 non-null     int64  
 1   WA Retail Gas Prices (Dollars per Gallon)  23 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 496.0 bytes
None
   Date  WA Retail Gas Prices (Dollars per Gallon)
0  2003                                      1.684
1  2004                                      1.991
2  2005                                      2.410
3  2006                                      2.751
4  2007                                      3.009


In [ ]:
df_gas.head()

## Electric Vehicle Population Dataset Cleaning

**Checking first few rows of dataset**

In [11]:
df_EVs.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,5UX43EU09S,Kitsap,Poulsbo,WA,98370.0,2025,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,40.0,23.0,267525737,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,5.303594e+10
4,3C3CFFGE5F,Thurston,Yelm,WA,98597.0,2015,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87.0,2.0,474468501,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10


**Dropping Duplicate Rows**

In [12]:
og_len = df_EVs.shape[0]
df_EVs.drop_duplicates()
new_len = df_EVs.shape[0]
print("Original Length:", og_len)
print("New Length:", new_len)

Original Length: 279780
New Length: 279780


**Checking for Uniqueness of each VIN number and DOL Vehicle ID**

In [13]:
print("Unique VIN Number Count:", df_EVs['VIN (1-10)'].nunique())
print("Unique DOL Vehicle ID Count:", df_EVs['DOL Vehicle ID'].nunique())
print("Total Original Table Length:", df_EVs.shape[0])

Unique VIN Number Count: 17072
Unique DOL Vehicle ID Count: 279780
Total Original Table Length: 279780


**Setting DOL Vehicle ID as table index**

In [14]:
df_EVs.set_index('DOL Vehicle ID', inplace=True)

**Dropping Unnecessary Columns**

In [15]:
df_EVs.drop(columns=['VIN (1-10)', 'Legislative District', '2020 Census Tract'], inplace=True)

**Changing Electric Vehicle Type column values to Plug-in Hybrid, Battery Electric, or None**

In [16]:
df_EVs.value_counts('Electric Vehicle Type')

Electric Vehicle Type
Battery Electric Vehicle (BEV)            223884
Plug-in Hybrid Electric Vehicle (PHEV)     55896
dtype: int64

In [17]:
df_EVs['Electric Vehicle Type'].replace({'Plug-in Hybrid Electric Vehicle (PHEV)': 'Plug-in Hybrid (PHEV)', 'Battery Electric Vehicle (BEV)': 'Battery Electric (BEV)'}, inplace=True)

**Changing the Clean Alternative Fuel Vehicle (CAFV) Eligibility to boolean values (True/False/None)**


In [18]:
df_EVs.value_counts('Clean Alternative Fuel Vehicle (CAFV) Eligibility')

Clean Alternative Fuel Vehicle (CAFV) Eligibility
Eligibility unknown as battery range has not been researched    177937
Clean Alternative Fuel Vehicle Eligible                          77495
Not eligible due to low battery range                            24348
dtype: int64

In [19]:
df_EVs['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].replace({
'Eligibility unknown as battery range has not been researched': None,
'Clean Alternative Fuel Vehicle Eligible': True,
'Not eligible due to low battery range': False,
}, inplace=True)

**Changing CAFV Eligibility column name to a question**

In [20]:
df_EVs.rename(columns={'Clean Alternative Fuel Vehicle (CAFV) Eligibility': 'Clean Alternative Fuel Vehicle (CAFV) Eligible?'}, inplace=True)

**Replacing 0.0 values in Electric Range to np.nan for cleaner averages**

In [21]:
df_EVs['Electric Range'] = df_EVs['Electric Range'].replace(0.0, np.nan)

**Changing Postal Code and Electric Ranges to Nullable Int64**

In [22]:
df_EVs['Electric Range'] = df_EVs['Electric Range'].astype('Int64')
df_EVs['Postal Code'] = df_EVs['Postal Code'].astype('Int64')

**Changing Column Name of "Electric Range" to "Electric Range (in Miles)"**

In [23]:
df_EVs.rename(columns={'Electric Range': 'Electric Range (in Miles)'}, inplace=True)
df_EVs.head()

,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligible?,Electric Range (in Miles),Vehicle Location,Electric Utility
DOL Vehicle ID,,,,,,,,,,,,
153331706,Stevens,Colville,WA,99114,2012,NISSAN,LEAF,Battery Electric (BEV),True,73,POINT (-117.90454 48.54657),AVISTA CORP
253586308,Yakima,Yakima,WA,98903,2023,SUBARU,SOLTERRA,Battery Electric (BEV),None,<NA>,POINT (-120.71847 46.55029),PACIFICORP
333135022,King,Seattle,WA,98122,2018,NISSAN,LEAF,Battery Electric (BEV),True,151,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA)
267525737,Kitsap,Poulsbo,WA,98370,2025,BMW,X5,Plug-in Hybrid (PHEV),True,40,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC
474468501,Thurston,Yelm,WA,98597,2015,FIAT,500,Battery Electric (BEV),True,87,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC


**Checking if Electric Utility values should be cleaned and deciding not to clean them**

In [24]:
df_EVs['Electric Utility'].unique()

array(['AVISTA CORP', 'PACIFICORP',
       'CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA)',
       'PUGET SOUND ENERGY INC',
       'PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)',
       'PUD NO 1 OF CHELAN COUNTY', 'PUD NO 2 OF GRANT COUNTY',
       'PUD NO 1 OF DOUGLAS COUNTY', 'MODERN ELECTRIC WATER COMPANY',
       'BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF CLARK COUNTY - (WA)',
       'CITY OF BLAINE - (WA)||PUD NO 1 OF WHATCOM COUNTY',
       'BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF MASON COUNTY|PUD NO 1 OF JEFFERSON COUNTY',
       'BONNEVILLE POWER ADMINISTRATION||PUD NO 2 OF PACIFIC COUNTY',
       'PUGET SOUND ENERGY INC||PUD NO 1 OF WHATCOM COUNTY',
       'BONNEVILLE POWER ADMINISTRATION||CITY OF TACOMA - (WA)||PENINSULA LIGHT COMPANY',
       'BONNEVILLE POWER ADMINISTRATION||CITY OF TACOMA - (WA)||ELMHURST MUTUAL POWER & LIGHT CO|PENINSULA LIGHT COMPANY',
       'BONNEVILLE POWER ADMINISTRATION||CITY OF TACOMA - (WA)||PARKLAND LIGHT & WATER COMPANY|PENINSULA LIGHT

**Checking data types in table and cleaned data**

In [ ]:
df_EVs.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 279780 entries, 153331706 to 267728625
Data columns (total 12 columns):
 #   Column                                           Non-Null Count   Dtype 
---  ------                                           --------------   ----- 
 0   County                                           279756 non-null  object
 1   City                                             279756 non-null  object
 2   State                                            279780 non-null  object
 3   Postal Code                                      279756 non-null  Int64 
 4   Model Year                                       279780 non-null  int64 
 5   Make                                             279780 non-null  object
 6   Model                                            279780 non-null  object
 7   Electric Vehicle Type                            279780 non-null  object
 8   Clean Alternative Fuel Vehicle (CAFV) Eligible?  101843 non-null  object
 9   Electric Range 

In [ ]:
df_EVs.head()

## Merging The Two Datasets

**Merging datasets so every car has a gas price associated with it.**

In [80]:
df_merged = df_EVs.merge(df_gas, how="left", left_on='Model Year', right_on='Date', validate=None)
df_merged.head()

,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligible?,Electric Range (in Miles),Vehicle Location,Electric Utility,Date,WA Retail Gas Prices (Dollars per Gallon)
0,Stevens,Colville,WA,99114,2012,NISSAN,LEAF,Battery Electric (BEV),True,73,POINT (-117.90454 48.54657),AVISTA CORP,2012.0,3.883
1,Yakima,Yakima,WA,98903,2023,SUBARU,SOLTERRA,Battery Electric (BEV),None,<NA>,POINT (-120.71847 46.55029),PACIFICORP,2023.0,4.541
2,King,Seattle,WA,98122,2018,NISSAN,LEAF,Battery Electric (BEV),True,151,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),2018.0,3.269
3,Kitsap,Poulsbo,WA,98370,2025,BMW,X5,Plug-in Hybrid (PHEV),True,40,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,2025.0,4.228
4,Thurston,Yelm,WA,98597,2015,FIAT,500,Battery Electric (BEV),True,87,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,2015.0,2.754


**Dropping duplicate date column**

In [82]:
df_merged.drop(columns=['Date'], inplace=True)
df_merged.head()

,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligible?,Electric Range (in Miles),Vehicle Location,Electric Utility,WA Retail Gas Prices (Dollars per Gallon)
0,Stevens,Colville,WA,99114,2012,NISSAN,LEAF,Battery Electric (BEV),True,73,POINT (-117.90454 48.54657),AVISTA CORP,3.883
1,Yakima,Yakima,WA,98903,2023,SUBARU,SOLTERRA,Battery Electric (BEV),None,<NA>,POINT (-120.71847 46.55029),PACIFICORP,4.541
2,King,Seattle,WA,98122,2018,NISSAN,LEAF,Battery Electric (BEV),True,151,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),3.269
3,Kitsap,Poulsbo,WA,98370,2025,BMW,X5,Plug-in Hybrid (PHEV),True,40,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,4.228
4,Thurston,Yelm,WA,98597,2015,FIAT,500,Battery Electric (BEV),True,87,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,2.754


<div style="display: flex; flex-direction: row; justify-content: space-between; font-weight: bold; text-decoration: none">
<a href="../README.md"><<< Back to README File (Introduction)</a>
<a href="02_exploratory_data_analysis.ipynb">Up Next: Exploratory Data Analysis (EDA) >>></a>
</div>